# Fact-check smoke eval

Runs the 18-question fact-check smoke set (`evals/eval_set.json`: facts, claim checks, multi-hop, ambiguous terms, out-of-scope) through `WikiQA` and shows each answer next to its expected outcome, with the tool trace.

Each expected answer is backed by a quoted sentence from a named Wikipedia section, pinned to a revision (`evals/verify_evidence.py` re-checks them). The larger fact-verification evals are `evals/run.py` / `evals/judge.py`; the research-brief eval is `evals/run_eval.py`.

In [ ]:
import json, sys, time
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wiki_search").is_dir())
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from wiki_search import WikiQA, wikipedia

In [ ]:
qa = WikiQA(model="claude-sonnet-4-6", effort="medium")
wikipedia.CACHE_MODE = "on"  # reruns read identical article text

In [ ]:
def tokens(a):
    u = a.usage
    return u["input_tokens"] + u["cache_read_input_tokens"] + u["cache_creation_input_tokens"], u["output_tokens"]


def expected_text(item):
    """One-line summary of an eval item's expected outcome."""
    if item["should_clarify"]:
        return "should ask for clarification" + (f" (or: {item['also_accept']})" if item.get("also_accept") else "")
    if item["category"] == "research":
        return "research brief covering: " + "; ".join(item["must_cover"])
    if item["category"] == "out-of-scope":
        return "should say Wikipedia doesn't cover this"
    parts = [item["expected_verdict"], item["expected_answer"]]
    note = " ⚖️ needs your decision" if item["review"]["status"] == "needs_decision" else ""
    return ": ".join(p for p in parts if p) + note


def render(answer, expected=None, item_id=None):
    """Show one answer: the response (or clarification request), expected answer, and the tool trace."""
    head = f"### {item_id + ': ' if item_id else ''}{answer.question}"
    parts = [head]
    if expected:
        parts.append(f"> **Expected:** {expected}")
    if answer.needs_clarification:
        c = answer.clarification
        opts = "\n".join(f"- {o}" for o in c.options)
        parts.append(f"**🔀 Needs clarification** (disambiguation page *{c.term}*)\n\n{c.question}\n\n{opts}\n\n_Why: {c.rationale}_")
    else:
        parts.append(answer.text)
        parts.extend(f"⚠️ {w}" for w in answer.warnings)
    tin, tout = tokens(answer)
    parts.append(f"<sub>{answer.elapsed_s}s · {answer.usage['llm_calls']} LLM calls · {tin:,} in / {tout:,} out tokens · stop: {answer.stop_reason}</sub>")
    display(Markdown("\n\n".join(parts)))

    steps = []
    for t in answer.trace:
        if t["type"] == "search":
            steps.append(f"🔎 <b>search</b> <code>{t['query']}</code> → {', '.join(t['results'][:5])}{' …' if len(t['results']) > 5 else ''}")
        elif t["type"] == "read":
            steps.append(f"📖 <b>read</b> {t['title']} § {t['section']}")
        elif t["type"] == "disambiguation":
            verdict = "ambiguous → ask user" if t["ambiguous"] else f"resolved → {t['most_likely_title']}"
            steps.append(f"🔀 <b>disambiguation</b> {t['page']}: {verdict}")
        elif t["type"] == "plan":
            steps.append("🗺️ <b>plan research</b>: " + "; ".join(t["subtopics"]))
        elif t["type"] == "revise":
            steps.append(f"✂️ <b>too long</b> ({t['words']} words): asked to shorten")
        elif t["type"] == "submit":
            steps.append("✅ <b>submit_answer</b>" + (f" ({len(t['warnings'])} warnings)" if t["warnings"] else ""))
        elif t["type"] == "error":
            steps.append(f"⚠️ <b>error</b> {t['tool']}: {t['error']}")
    if steps:
        display(HTML("<details><summary>Tool trace (" + str(len(steps)) + " steps)</summary><ol>"
                     + "".join(f"<li>{s}</li>" for s in steps) + "</ol></details><hr>"))

In [ ]:
eval_set = json.loads((ROOT / "evals" / "eval_set.json").read_text())

# Sequential on purpose: Wikimedia asks clients not to send parallel requests.
wikipedia.stats.clear()
start = time.time()
answers = []
for item in eval_set:
    answers.append(qa.ask(item["question"]))
    print(f"{item['id']}: {answers[-1].elapsed_s}s")
print(f"\nRan {len(eval_set)} questions in {time.time() - start:.0f}s. "
      f"Wikipedia: {wikipedia.stats['requests']} requests, {wikipedia.stats['cache_hits']} cache hits.")

## Summary

In [ ]:
rows = []
for item, a in zip(eval_set, answers):
    tin, tout = tokens(a)
    searches = sum(t["type"] == "search" for t in a.trace)
    reads = sum(t["type"] == "read" for t in a.trace)
    outcome = "clarify" if a.needs_clarification else a.stop_reason
    s = a.structured or {}
    short = " ".join(filter(None, [s.get("verdict"), s.get("answer", "")]))[:60]
    rows.append((item["id"], item["category"], outcome, short, searches, reads, a.elapsed_s, tin, tout))

header = ["id", "category", "outcome", "answer", "searches", "reads", "seconds", "tokens in", "tokens out"]
table = "| " + " | ".join(header) + " |\n|" + "---|" * len(header) + "\n"
table += "\n".join("| " + " | ".join(f"{v:,}" if isinstance(v, int) else str(v) for v in r) + " |" for r in rows)
tot_in, tot_out = sum(r[7] for r in rows), sum(r[8] for r in rows)
# Sonnet 4.6 list price: $3 / $15 per million tokens (cache reads are billed lower, so this is an upper bound).
table += f"\n\n**Total:** {tot_in:,} in / {tot_out:,} out tokens ≈ ${tot_in * 3e-6 + tot_out * 15e-6:.2f} (upper bound)"
display(Markdown(table))

## Answers

In [ ]:
for item, a in zip(eval_set, answers):
    render(a, expected=expected_text(item), item_id=item["id"])